## I. Tiền Xử Lý Dữ Liệu Spam SMS
Pipeline gồm 7 bước: Xoá duplicate → Encode label → Làm sạch văn bản → Tokenize + Stopwords → Train/Test split → Vector hoá (TF-IDF) → Balance Dataset (SMOTE)

### Cài đặt thư viện

In [2]:
# Chạy cell này một lần để cài đặt
!pip install pandas scikit-learn nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\hary0\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Import thư viện

In [3]:
import pandas as pd # xử lý dữ liệu dạng bảng
import numpy as np # xử lý mảng, ma trận
import re # xử lý text (regex)
import pickle # lưu/truy xuất file nhị phân
import nltk # xử lý ngôn ngữ tự nhiên
from nltk.corpus import stopwords # loại bỏ stopwords (từ dừng)
from sklearn.model_selection import train_test_split # chia tập train/test
from sklearn.feature_extraction.text import TfidfVectorizer # trích xuất đặc trưng TF-IDF
from scipy.sparse import save_npz # lưu ma trận thưa dưới định dạng .npz
from imblearn.over_sampling import SMOTE # balance dataset

nltk.download('stopwords')

print('Import xong!')

Import xong!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hary0\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Load dữ liệu gốc

In [4]:
from pathlib import Path

csv_paths = [
    Path('spam.csv'),
    Path('../data/spam.csv'),
    Path('../spam.csv'),
    Path('../../data/spam.csv')
]

for p in csv_paths:
    if p.exists():
        csv_file = p
        break
else:
    raise FileNotFoundError('Không tìm thấy file spam.csv trong thư mục hiện tại, data/, hoặc thư mục cha.')

# Thử tự phát hiện encoding nếu cần
for enc in ['utf-8', 'latin-1', 'ISO-8859-1', 'cp1252']:
    try:
        df = pd.read_csv(csv_file, encoding=enc)
        print(f'Loaded {csv_file} bằng encoding {enc}')
        break
    except Exception as e:
        last_err = e
else:
    raise last_err

# Nếu cột không chuẩn (vì spam.csv trên internet có cột khác), fix tên
if 'v1' in df.columns and 'v2' in df.columns:
    df = df.rename(columns={'v1': 'label', 'v2': 'message'})

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head()

Loaded ..\data\spam.csv bằng encoding utf-8
Shape: (5572, 2)
Columns: ['Category', 'Message']



,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


---
### Bước 1 — Delete Duplicate
Loại bỏ các dòng trùng lặp để tránh **data leakage** khi chia tập train/test.

In [5]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)
print(f'Trước : {before} dòng')
print(f'Sau   : {after} dòng')
print(f'Đã xoá: {before - after} dòng trùng')

Trước : 5572 dòng
Sau   : 5157 dòng
Đã xoá: 415 dòng trùng


---
### Bước 2 — Encode Label 
Chuyển nhãn dạng text sang số: `ham → 0`, `spam → 1`.

In [6]:
df['label'] = df['Category'].map({'ham': 0, 'spam': 1})

print('Phân phối nhãn sau encode:')
print(df['label'].value_counts())
print()
print(f'Null values: {df["label"].isnull().sum()}')
df[['Category', 'label']].head(6)

Phân phối nhãn sau encode:
label
0    4516
1     641
Name: count, dtype: int64

Null values: 0


,Category,label
0,ham,0
1,ham,0
2,spam,1
3,ham,0
4,ham,0
5,spam,1


---
### Bước 3 — Text Cleaning
Chuẩn hoá text: lowercase, xoá URL, số, ký tự đặc biệt và khoảng trắng thừa.

In [7]:
def clean_text(text):
    text = str(text).lower()                          # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)       # xoá URL
    text = re.sub(r'\d+', '', text)                   # xoá số
    text = re.sub(r'[^a-z\s]', '', text)              # xoá ký tự đặc biệt
    text = re.sub(r'\s+', ' ', text).strip()          # xoá khoảng trắng thừa
    return text

df['clean_msg'] = df['Message'].apply(clean_text)

# Kiểm tra kết quả
print('Ví dụ trước/sau làm sạch:\n')
for i in [2, 5, 10]:
    print(f'  TRƯỚC: {df["Message"].iloc[i][:90]}')
    print(f'  SAU  : {df["clean_msg"].iloc[i][:90]}')
    print()

Ví dụ trước/sau làm sạch:

  TRƯỚC: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to re
  SAU  : free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry ques

  TRƯỚC: FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you u
  SAU  : freemsg hey there darling its been weeks now and no word back id like some fun you up for 

  TRƯỚC: I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k? I've 
  SAU  : im gonna be home soon and i dont want to talk about this stuff anymore tonight k ive cried



---
### Bước 4 — Tokenize + Loại Stopwords
Tách từ và loại bỏ các từ phổ biến không mang nhiều ý nghĩa (the, is, a, to...).

In [8]:
stop_words = set(stopwords.words('english'))

def tokenize_and_remove_stopwords(text):
    tokens = str(text).split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

df['processed'] = df['clean_msg'].apply(tokenize_and_remove_stopwords)

# Thống kê
avg_before = df['clean_msg'].str.split().apply(len).mean()
avg_after  = df['processed'].str.split().apply(len).mean()
print(f'Độ dài TB trước: {avg_before:.1f} từ/tin nhắn')
print(f'Độ dài TB sau  : {avg_after:.1f} từ/tin nhắn')
print()

# Xem ví dụ
print('Ví dụ trước/sau tokenize:\n')
for i in [0, 2, 10]:
    print(f'  TRƯỚC: {df["clean_msg"].iloc[i][:90]}')
    print(f'  SAU  : {df["processed"].iloc[i][:90]}')
    print()

Độ dài TB trước: 14.7 từ/tin nhắn
Độ dài TB sau  : 8.4 từ/tin nhắn

Ví dụ trước/sau tokenize:

  TRƯỚC: go until jurong point crazy available only in bugis n great world la e buffet cine there g
  SAU  : go jurong point crazy available bugis great world la buffet cine got amore wat

  TRƯỚC: free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry ques
  SAU  : free entry wkly comp win fa cup final tkts st may text fa receive entry questionstd txt ra

  TRƯỚC: im gonna be home soon and i dont want to talk about this stuff anymore tonight k ive cried
  SAU  : im gonna home soon dont want talk stuff anymore tonight ive cried enough today



---
### Bước 5 — Train / Test Split
Chia 80/20 với `stratify=label` để giữ nguyên tỉ lệ lớp ham/spam trong cả hai tập.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df['processed'],
    df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print(f'Train set : {len(X_train)} mẫu')
print(f'Test set  : {len(X_test)} mẫu')
print()
print('Tỉ lệ lớp trong train:')
print(y_train.value_counts(normalize=True).round(3))
print()
print('Tỉ lệ lớp trong test:')
print(y_test.value_counts(normalize=True).round(3))

Train set : 4125 mẫu
Test set  : 1032 mẫu

Tỉ lệ lớp trong train:
label
0    0.876
1    0.124
Name: proportion, dtype: float64

Tỉ lệ lớp trong test:
label
0    0.876
1    0.124
Name: proportion, dtype: float64


---
### Bước 6 — Vector Hoá (TF-IDF)
Chuyển văn bản thành vector số.

> **Quan trọng:** Chỉ `fit_transform` trên tập **train**, tập **test** chỉ dùng `transform` — tránh data leakage.

In [10]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)   # fit + transform trên train
X_test_vec  = vectorizer.transform(X_test)        # chỉ transform trên test

print(f'Shape X_train_vec : {X_train_vec.shape}')
print(f'Shape X_test_vec  : {X_test_vec.shape}')

Shape X_train_vec : (4125, 5000)
Shape X_test_vec  : (1032, 5000)


### Bước 7 — Balance Dataset (SMOTE)
Dataset hiện tại imbalance (ham 87.6%, spam 12.4%). Dùng SMOTE để oversample lớp thiểu số (spam).

In [11]:
# Áp dụng SMOTE chỉ trên train set để tránh data leakage
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_vec, y_train)

print(f'Trước balance: X_train {X_train_vec.shape}, y_train ham: {int((y_train==0).sum())}, spam: {int((y_train==1).sum())}')
print(f'Sau balance: X_train {X_train_balanced.shape}, y_train ham: {int((y_train_balanced==0).sum())}, spam: {int((y_train_balanced==1).sum())}')

# Cập nhật X_train_vec và y_train
X_train_vec = X_train_balanced
y_train = y_train_balanced
print(f'Số features (từ)  : {len(vectorizer.vocabulary_)}')

# Top 20 từ theo TF-IDF trung bình
feature_names = vectorizer.get_feature_names_out()
mean_tfidf    = np.asarray(X_train_vec.mean(axis=0)).flatten()
top20_idx     = mean_tfidf.argsort()[-20:][::-1]
print()
print('Top 20 từ có TF-IDF cao nhất:')
print([feature_names[i] for i in top20_idx])

Trước balance: X_train (4125, 5000), y_train ham: 3612, spam: 513
Sau balance: X_train (7224, 5000), y_train ham: 3612, spam: 3612
Số features (từ)  : 5000

Top 20 từ có TF-IDF cao nhất:
['call', 'free', 'ur', 'txt', 'text', 'mobile', 'claim', 'stop', 'get', 'prize', 'reply', 'urgent', 'new', 'please', 'contact', 'im', 'send', 'ok', 'cash', 'week']


### Lưu kết quả

In [12]:
# Lưu dataframe đã xử lý
df.to_csv('../data/spam_processed.csv', index=False)

# Lưu train/test set dạng CSV
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df  = pd.DataFrame({'text': X_test,  'label': y_test})
train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv',   index=False)

# Lưu TF-IDF matrix
save_npz('../data/X_train.npz', X_train_vec)
save_npz('../data/X_test.npz',  X_test_vec)
np.save('../data/y_train.npy', y_train.values)
np.save('../data/y_test.npy',  y_test.values)

# Lưu vectorizer để dùng lại khi predict
with open('../data/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print(' Đã lưu dữ liệu đã xử lý và balance vào ../data/')
print('   ../data/spam_processed.csv   — toàn bộ dataframe đã xử lý')
print('   ../data/train.csv / ../data/test.csv — tập train và test')
print('   ../data/X_train.npz / ../data/X_test.npz — ma trận TF-IDF')
print('   ../data/y_train.npy / ../data/y_test.npy — nhãn')
print('   ../data/tfidf_vectorizer.pkl — vectorizer (dùng khi predict)')

 Đã lưu dữ liệu đã xử lý và balance vào ../data/
   ../data/spam_processed.csv   — toàn bộ dataframe đã xử lý
   ../data/train.csv / ../data/test.csv — tập train và test
   ../data/X_train.npz / ../data/X_test.npz — ma trận TF-IDF
   ../data/y_train.npy / ../data/y_test.npy — nhãn
   ../data/tfidf_vectorizer.pkl — vectorizer (dùng khi predict)
